# 4. 선형 StateGraph 기초

**시나리오:** 질문이 검색·답변·근거 검증 node를 순서대로 통과합니다.

**학습 목표:** `QueryState`, `StateGraph`, node, edge와 누적 `steps` trace의 관계를 익힙니다.

## 중요 변수·함수

- `build_workflow()`: `START → retrieve → answer → grounding_guard → END` graph를 compile합니다.
- `QueryState.documents`: retrieve node가 소유하는 근거입니다.
- `steps`: 각 node가 자신의 실행 흔적만 추가하는 누적 필드입니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# compiled graph에 최소 초기 state를 전달합니다.
from week1.app import build_workflow, create_fixture_services

graph = build_workflow(create_fixture_services())
state = graph.invoke({'question': 'vacation request notice', 'steps': []})
state['steps']

In [ ]:
# 최종 state의 근거와 상태가 trace와 일치하는지 확인합니다.
assert state['steps'] == ['retrieve', 'answer', 'grounding_guard']
assert state['status'] == 'answered'
{'status': state['status'], 'citations': state['citations']}

## 예측 과제와 해석

**예측 과제:** retrieve가 빈 목록을 반환할 때 `answer`와 `grounding_guard`가 어떤 값을 남길지 예상하세요.

**해석:** node가 전체 state를 다시 만들지 않고 자신이 책임지는 필드만 갱신하면 흐름과 실패 지점을 추적하기 쉬워집니다.